# 10. Does the stone sound like a bell?

Quantitative comparison between the Piedra Campana and a metallic, bell-like resonator recorded with the same chain
(stainless-steel bowl, notebook 09), and a test of whether the near-consonant intervals between the modes of the stone
can be a coincidence.

**Inputs:** `tables/t06_mode_table.csv`, `tables/t09_bowl_T60.csv`, WAV files of the bowl (`C.INSTRUMENT_CHECK`).
**Outputs:** `tables/t10_bowl_partials.csv`, `tables/t10_intervals.csv`.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # folder with config.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config as C
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.4})
pd.set_option("display.precision", 2)

from scipy import signal
from scipy.io import wavfile
warnings.filterwarnings("ignore", category=wavfile.WavFileWarning)
FS = C.FS

## 10.1 Partials of the bowl

Spectrum of 1 s, from 30 ms after one strike (microphone at 5 cm), and peaks with a prominence ≥ 15 dB.

In [2]:
_, x = wavfile.read(C.INSTRUMENT_CHECK / "20261028-233714_5cm.wav"); x = x.astype(float)
b, a = signal.butter(4, 2000 / (FS / 2), "high"); env = np.convolve(np.abs(signal.filtfilt(b, a, x)), np.ones(96) / 96, "same")
p, _ = signal.find_peaks(env, height=np.median(env) * 30, distance=int(3 * FS)); i = p[2]
s = x[i + int(0.03 * FS): i + int(1.03 * FS)]; N = 2 ** 19
X = np.abs(np.fft.rfft(s * np.hanning(len(s)), N)); fr = np.fft.rfftfreq(N, 1 / FS); m = (fr > 100) & (fr < 10000)
Xd = 20 * np.log10(X / X[m].max() + 1e-12)
q, _ = signal.find_peaks(np.where(m, Xd, -200), prominence=15, distance=int(8 / fr[1]), height=-60)
bowl = pd.DataFrame({"f_Hz": fr[q], "level_dB": Xd[q], "ratio": fr[q] / fr[q][np.argmax(Xd[q])]})
bowl.to_csv(C.TABLES / "t10_bowl_partials.csv", index=False)
bowl.round(3)

,f_Hz,level_dB,ratio
0,352.48,0.00,1.0
1,704.96,-49.40,2.0
2,986.48,-41.13,2.8


## 10.2 Damping: stone against bowl

Q of the main mode of the bowl (22 strikes) against the median Q of each mode of the stone.

In [3]:
tb = pd.read_csv(C.TABLES / "t09_bowl_T60.csv"); tm = pd.read_csv(C.TABLES / "t06_mode_table.csv")
Qb = tb.Q.median()
print(f"Bowl (352.5 Hz): Q = {Qb:.0f} (range {tb.Q.min():.0f}–{tb.Q.max():.0f}), T60 = {tb.T60_s.median():.1f} s")
print(f"Stone: Q from {tm.Q_median.min():.0f} to {tm.Q_median.max():.0f}; {(tm.Q_median >= 1000).sum()} of {len(tm)} modes with Q ≥ 1000")
print("T60 the stone would have at 352.5 Hz with its median Q:",
      f"{tm.Q_median.median() * np.log(1000) / (np.pi * 352.5):.1f} s (T60 = Q·ln1000/(π f))")

Bowl (352.5 Hz): Q = 1241 (range 1109–1349), T60 = 7.7 s
Stone: Q from 378 to 1237; 9 of 13 modes with Q ≥ 1000
T60 the stone would have at 352.5 Hz with its median Q: 6.5 s (T60 = Q·ln1000/(π f))


## 10.3 Intervals between the modes of the stone: coincidence?

The pairs among the 13 modes whose interval lies within 25 cents of an octave (1200 cents), fifth (702), fourth (498),
major third (386) or minor third (316) are counted. The value expected by chance is estimated with 20 000 sets of 13
frequencies uniformly distributed on a logarithmic scale between 385 and 1938 Hz.

In [4]:
TARGET = {"octave": 1200, "fifth": 702, "fourth": 498, "major third": 386, "minor third": 316}
def pairs(f, tol=25):
    f = np.sort(f); out = []
    for i in range(len(f)):
        for j in range(i + 1, len(f)):
            c = 1200 * np.log2(f[j] / f[i])
            for k, v in TARGET.items():
                if abs(c - v) <= tol: out.append((f[i], f[j], k, c - v))
    return out
fm = tm.f_mean_Hz.values
obs = pairs(fm)
tab = pd.DataFrame(obs, columns=["f1_Hz", "f2_Hz", "interval", "deviation_cents"]); tab.to_csv(C.TABLES / "t10_intervals.csv", index=False)
rng = np.random.default_rng(C.SEED)
sim = np.array([len(pairs(np.exp(rng.uniform(np.log(385), np.log(1938), len(fm))))) for _ in range(20000)])
print(f"Pairs within ±25 cents of a consonance: observed {len(obs)}; expected by chance {sim.mean():.1f} "
      f"(95th percentile: {np.percentile(sim, 95):.0f}); p = {(sim >= len(obs)).mean():.2f}")
tab.round(1)

Pairs within ±25 cents of a consonance: observed 10; expected by chance 10.8 (95th percentile: 17); p = 0.64


,f1_Hz,f2_Hz,interval,deviation_cents
0,384.9,759.2,octave,-23.9
1,599.4,759.2,major third,23.0
2,848.8,1260.0,fifth,-18.1
3,969.5,1298.8,fourth,8.2
4,969.5,1938.0,octave,-0.9
5,1260.0,1893.3,fifth,3.0
6,1298.8,1645.3,major third,23.4
7,1298.8,1938.0,fifth,-9.1
8,1348.8,1819.4,fourth,20.2
9,1389.2,1645.3,minor third,-23.2


## 10.4 Conclusion

* **Damping:** the least damped modes of the stone have a Q of the same order as the main mode of a steel bowl
  recorded with the same chain. In that sense the stone behaves like a metallic resonator; it rings for a shorter time
  than the bowl because its modes are higher (T60 = Q·ln 1000/(π f)).
* **Spectrum:** the bowl, struck with the same hammer, sounds almost like a pure tone (its second partial is ~40 dB
  weaker), whereas the stone spreads its energy over several strong, inharmonic partials. The complex, "metallic"
  character of the stone is closer to that of a bell than to that of a bowl.
* **Intervals:** there are 10 near-consonant pairs among the 13 modes, and chance produces 10.8 on average (p = 0.64).
  The near-octaves (385–759 and 970–1938 Hz) and the near-major third (599–759 Hz) are **numerical coincidences**: the
  stone is not "tuned".